## Retraining from Scratch

In [10]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import Subset, DataLoader

from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
from tqdm import tqdm
import numpy as np
import time
import copy

In [11]:
def get_mnist_data(delete_class=7):
    transform = transforms.Compose([transforms.ToTensor()])
    train_set = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=transform)
    test_set = torchvision.datasets.MNIST(root='./data', train=False, download=True, transform=transform)

    # Indices for class to delete
    deleted_indices = [i for i, (_, label) in enumerate(train_set) if label == delete_class]
    retained_indices = [i for i in range(len(train_set)) if i not in deleted_indices]

    # Subsets
    full_loader = torch.utils.data.DataLoader(train_set, batch_size=64, shuffle=True)
    unlearn_loader = torch.utils.data.DataLoader(Subset(train_set, retained_indices), batch_size=64, shuffle=True)
    test_loader = torch.utils.data.DataLoader(test_set, batch_size=64, shuffle=False)

    # Test samples with deleted class only
    test_deleted_class = [i for i, (_, label) in enumerate(test_set) if label == delete_class]
    test_deleted_loader = DataLoader(Subset(test_set, test_deleted_class), batch_size=64, shuffle=False)

    return train_set, test_set, full_loader, unlearn_loader, test_loader, test_deleted_loader, deleted_indices

In [14]:
class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )
        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32 * 7 * 7, 128),
            nn.ReLU(),
            nn.Linear(128, 10)
        )

    def forward(self, x):
        return self.fc(self.conv(x))


In [15]:
def train_cnn(model, train_loader, epochs=3):
    model.train()
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    criterion = nn.CrossEntropyLoss()

    for epoch in range(epochs):
        for imgs, labels in train_loader:
            optimizer.zero_grad()
            output = model(imgs)
            loss = criterion(output, labels)
            loss.backward()
            optimizer.step()
    return model

def test_cnn(model, test_loader):
    model.eval()
    correct = total = 0
    with torch.no_grad():
        for imgs, labels in test_loader:
            outputs = model(imgs)
            pred = outputs.argmax(1)
            correct += (pred == labels).sum().item()
            total += labels.size(0)
    return correct / total

def model_distance(model1, model2):
    vec1 = torch.cat([p.view(-1) for p in model1.parameters()])
    vec2 = torch.cat([p.view(-1) for p in model2.parameters()])
    return torch.norm(vec1 - vec2).item()

In [12]:
def decision_tree_train(train_set, indices=None):
    X = []
    y = []
    for i in (indices if indices is not None else range(len(train_set))):
        img, label = train_set[i]
        X.append(img.view(-1).numpy())
        y.append(label)
    clf = DecisionTreeClassifier()
    clf.fit(X, y)
    return clf

def decision_tree_test(model, test_set):
    X_test = [img.view(-1).numpy() for img, _ in test_set]
    y_true = [label for _, label in test_set]
    y_pred = model.predict(X_test)
    return accuracy_score(y_true, y_pred)

def decision_tree_test_on_deleted(model, train_set, deleted_indices):
    X = []
    y_true = []
    for idx in deleted_indices:
        img, label = train_set[idx]
        X.append(img.view(-1).numpy())
        y_true.append(label)
    y_pred = model.predict(X)
    return accuracy_score(y_true, y_pred)

In [13]:
DELETE_CLASS = 7
print(f"Loading data (Deleting class {DELETE_CLASS})...")
train_set, test_set, full_loader, unlearn_loader, test_loader, test_deleted_loader, deleted_indices = get_mnist_data(delete_class=DELETE_CLASS)

### ---- Decision Tree ---- ###
print("\n[Decision Tree] Training on full data...")
dt_full = decision_tree_train(train_set)

acc_full = decision_tree_test(dt_full, test_set)
acc_full_deleted = decision_tree_test(dt_full, Subset(test_set, [i for i, (_, l) in enumerate(test_set) if l == DELETE_CLASS]))

print("[Decision Tree] Retraining without deleted class...")
start = time.time()

retained_indices = [i for i in range(len(train_set)) if i not in deleted_indices]
dt_unlearned = decision_tree_train(train_set, indices=retained_indices)
end = time.time()
dt_unlearning_time = end - start

acc_unlearn = decision_tree_test(dt_unlearned, test_set)
acc_unlearn_deleted = decision_tree_test(dt_unlearned, Subset(test_set, [i for i, (_, l) in enumerate(test_set) if l == DELETE_CLASS]))

print(f"[Decision Tree] Accuracy Before: {acc_full:.4f}, After: {acc_unlearn:.4f}, Time: {dt_unlearning_time:.2f}s")
print(f"[Decision Tree] Deleted Class {DELETE_CLASS} Accuracy Before: {acc_full_deleted:.4f}, After: {acc_unlearn_deleted:.4f}")


Loading data (Deleting class 7)...

[Decision Tree] Training on full data...
[Decision Tree] Retraining without deleted class...
[Decision Tree] Accuracy Before: 0.8763, After: 0.7892, Time: 37.57s
[Decision Tree] Deleted Class 7 Accuracy Before: 0.8988, After: 0.0000


In [16]:
print("\n[CNN] Training on full data...")
cnn = CNN()
cnn_full = train_cnn(copy.deepcopy(cnn), full_loader)
acc_cnn_full = test_cnn(cnn_full, test_loader)
acc_cnn_full_deleted = test_cnn(cnn_full, test_deleted_loader)

print("[CNN] Retraining without deleted class...")

start = time.time()
cnn_unlearned = train_cnn(copy.deepcopy(cnn), unlearn_loader)
end = time.time()
cnn_unlearning_time = end - start

acc_cnn_unlearn = test_cnn(cnn_unlearned, test_loader)
acc_cnn_unlearn_deleted = test_cnn(cnn_unlearned, test_deleted_loader)

cnn_distance = model_distance(cnn_full, cnn_unlearned)

print(f"[CNN] Accuracy Before: {acc_cnn_full:.4f}, After: {acc_cnn_unlearn:.4f}, Time: {cnn_unlearning_time:.2f}s")
print(f"[CNN] Deleted Class {DELETE_CLASS} Accuracy Before: {acc_cnn_full_deleted:.4f}, After: {acc_cnn_unlearn_deleted:.4f}")
print(f"[CNN] Weight Distance Between Models: {cnn_distance:.4f}")


[CNN] Training on full data...
[CNN] Retraining without deleted class...
[CNN] Accuracy Before: 0.9859, After: 0.8888, Time: 108.47s
[CNN] Deleted Class 7 Accuracy Before: 0.9912, After: 0.0000
[CNN] Weight Distance Between Models: 15.6051


## SISA: Shared, Isloated, Slicing, Aggregation

In [17]:
import numpy as np
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
from torchvision import datasets, transforms
import random
from collections import Counter

In [18]:
def load_mnist_flat():
    transform = transforms.Compose([transforms.ToTensor()])
    trainset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
    testset = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

    X_train = [np.array(img[0]).reshape(-1).astype(np.float32) for img in trainset]
    y_train = [label for _, label in trainset]
    X_test = [np.array(img[0]).reshape(-1).astype(np.float32) for img in testset]
    y_test = [label for _, label in testset]

    return np.array(X_train), np.array(y_train), np.array(X_test), np.array(y_test)

In [19]:
# ----- SISA TRAINING -----
def sisa_train_decision_tree(X, y, num_shards=5, num_slices=3, epochs_per_slice=1):
    shard_models = []
    shard_slices = []
    data = list(zip(X, y))
    random.shuffle(data)
    shard_size = len(data) // num_shards

    for i in range(num_shards):
        shard_data = data[i * shard_size: (i + 1) * shard_size]
        slice_size = len(shard_data) // num_slices
        slices = []

        model = None
        for s in range(num_slices):
            current_slice = shard_data[: (s + 1) * slice_size]
            X_slice = [x for x, _ in current_slice]
            y_slice = [y for _, y in current_slice]

            if model==None:
              model = DecisionTreeClassifier()
            else:
              model = slices[-1]

            model.fit(X_slice, y_slice)
            slices.append(model)

        shard_slices.append(slices)
        shard_models.append(slices[-1])  # Final model for shard

    return shard_slices, shard_models

In [20]:
def sisa_predict(models, X):
    predictions = []
    for x in X:
        votes = [m.predict(x.reshape(1, -1))[0] for m in models]
        most_common = Counter(votes).most_common(1)[0][0]
        predictions.append(most_common)
    return np.array(predictions)

In [31]:
# ----- Unlearning -----
def sisa_unlearn_decision_tree(shard_slices, X, y, class_to_remove=7,num_shards=5, num_slices=3):
    new_shard_models = []
    shard_size = len(y) // num_shards
    for shard_i,shard in enumerate(shard_slices):
        # Find earliest slice affected

        affected_idx = None
        for i, model in enumerate(shard):
            # Reconstruct training data for this slice
            shard_y = y[i * shard_size: (i + 1) * shard_size]
            slice_size = len(y) // num_slices
            y_slice = shard_y[: (i + 1) * slice_size]

            if class_to_remove in y_slice:
                affected_idx = i
                break

        if affected_idx is None:
            new_shard_models.append(shard[-1])
            continue

        else:
          print(shard_i,end="=>")
          # Rebuild model from earliest affected slice
          new_slices = []
          for i in range(affected_idx):
            new_slices.append(shard[i])

          # Get data
          for i in range(affected_idx,num_slices):
            slice_size = len(y) // num_slices

            shard_X = X[i * shard_size: (i + 1) * shard_size]
            shard_y = y[i * shard_size: (i + 1) * shard_size]

            data_x = shard_X[: (i + 1) * slice_size]
            data_y = shard_y[: (i + 1) * slice_size]
            mask_not_7 = (data_y != 7)

            if len(new_slices)==0:
              model = DecisionTreeClassifier()
              model.fit(data_x[mask_not_7], data_y[mask_not_7])
              new_slices.append(model)

            else:
              model = new_slices[-1]
              model.fit(data_x[mask_not_7], data_y[mask_not_7])
              new_slices.append(model)

            new_shard_models.append(new_slices[-1])

    return new_shard_models

In [26]:
X_train, y_train, X_test, y_test = load_mnist_flat()

In [32]:
print("[Decision Tree] SISA Training on full data...")
slices, models = sisa_train_decision_tree(X_train, y_train)
y_pred_before = sisa_predict(models, X_test)
acc_before = accuracy_score(y_test, y_pred_before)

# Unlearn Class 7
print("[Decision Tree] SISA Unlearning...")
start  = time.time()
models_unlearn = sisa_unlearn_decision_tree(slices, X_train, y_train, class_to_remove=7)
end = time.time()
dt_unlearning_time = end - start

y_pred_after = sisa_predict(models_unlearn, X_test)
acc_after = accuracy_score(y_test, y_pred_after)

# Accuracy specifically on class 7 before unlearning
mask_7 = (y_test == 7)
acc_before_class7 = accuracy_score(y_test[mask_7], y_pred_before[mask_7])
acc_after_class7 = accuracy_score(y_test[mask_7], y_pred_after[mask_7])

print(f"[Decision Tree] Accuracy Before: {acc_before:.4f}, After: {acc_after:.4f}, Time: {dt_unlearning_time:.2f}s")
print(f"[Decision Tree] Accuracy on Class 7 Before: {acc_before_class7:.4f}, After: {acc_after_class7:.4f}")

[Decision Tree] SISA Training on full data...
[Decision Tree] SISA Unlearning...
0=>1=>2=>3=>4=>[Decision Tree] Accuracy Before: 0.9036, After: 0.7492, Time: 50.31s
[Decision Tree] Accuracy on Class 7 Before: 0.9193, After: 0.0000
